In [1]:
from datasets import load_dataset
CSV_FILE_PATH = r"C:\Users\choud\Downloads\wiki_hybrid_chunks_300.csv"
SFT_MODEL_DIR = "./sft_tinyllama"
dataset = load_dataset("csv", data_files=CSV_FILE_PATH)


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)


In [3]:
print(dataset.column_names)


{'train': ['group_id', 'section', 'chunk_id', 'chunk_text']}


In [4]:
def format_example(example):
    return {
        "text": f"<|user|> Summarize the following:\n{example['chunk_text']} <|assistant|>"
    }


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


In [6]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(MODEL_ID)


In [8]:
def tokenize_function(examples):
    return tokenizer(examples["chunk_text"], truncation=True, padding="max_length")


In [9]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)


In [10]:
import transformers
print(transformers.__version__)


4.52.4


In [11]:
from transformers import TrainingArguments
print(TrainingArguments.__module__)


transformers.training_args


In [12]:
from transformers import TrainingArguments

try:
    training_args = TrainingArguments(output_dir="./test_dir")
    print("TrainingArguments created with minimal args successfully")
except Exception as e:
    print("Error:", e)


TrainingArguments created with minimal args successfully


In [13]:
import inspect
from transformers import TrainingArguments

print(inspect.signature(TrainingArguments.__init__))


(self, output_dir: Optional[str] = None, overwrite_output_dir: bool = False, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: Union[transformers.trainer_utils.IntervalStrategy, str] = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, per_gpu_train_batch_size: Optional[int] = None, per_gpu_eval_batch_size: Optional[int] = None, gradient_accumulation_steps: int = 1, eval_accumulation_steps: Optional[int] = None, eval_delay: Optional[float] = 0, torch_empty_cache_steps: Optional[int] = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: Union[transformers.trainer_utils.SchedulerType, str] = 'linear', lr_scheduler_kwargs: Union[dict, str, NoneType] = <factory>, warmup_ratio: float = 0.0, warmup_ste

In [14]:
print(tokenized_dataset.keys())


dict_keys(['train'])


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling

# 1. Load raw dataset 
dataset = load_dataset("csv", data_files=CSV_FILE_PATH, split="train")

# Checking the columns of dataset before proceeding:
print(dataset.column_names)

# 2. Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

# 3. Tokenize dataset
def tokenize_function(examples):
    return tokenizer(examples["chunk_text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(
    tokenize_function, 
    batched=True, 
    remove_columns=dataset.column_names
)

# 4. Split train dataset into train + validation
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)
# split_dataset keys: 'train' and 'test' (use 'test' as validation)

# 5. Setup TrainingArguments
training_args = TrainingArguments(
    output_dir=SFT_MODEL_DIR,
    per_device_train_batch_size=1,  
    evaluation_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    num_train_epochs=3,
    fp16=False,  
    save_total_limit=2,
    logging_dir=SFT_MODEL_DIR,
    dataloader_pin_memory=False  # Optional: avoids warning if using CPU
)

# 6. Data collator for causal LM (no masking)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 7. Trainer initialization
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    tokenizer=tokenizer,  # safe to leave for now despite deprecation warning
    data_collator=data_collator,
)

# 8. Start training
trainer.train()

# 9. Save model and tokenizer
model.save_pretrained(SFT_MODEL_DIR)
tokenizer.save_pretrained(SFT_MODEL_DIR)


In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("No GPU detected, training will use CPU and will be slow.")


In [3]:
from datasets import load_dataset


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling

dataset = load_dataset("csv", data_files=CSV_FILE_PATH, split="train")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

def tokenize_function(examples):
    return tokenizer(examples["chunk_text"], truncation=True, max_length=128, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset["train"].column_names)

split_dataset = tokenized_dataset["train"].train_test_split(test_size=0.1)

training_args = TrainingArguments(
    output_dir=SFT_MODEL_DIR,
    per_device_train_batch_size=1,
    num_train_epochs=3,
    logging_steps=100,
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    save_total_limit=2,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()
model.save_pretrained(SFT_MODEL_DIR)
tokenizer.save_pretrained(SFT_MODEL_DIR)


C:\Users\gadep\AppData\Local\Temp\ipykernel_16720\3047035269.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


In [12]:
print(reward_dataset["train"].column_names)


['group_id', 'section', 'chunk_id', 'chunk_text']


In [5]:
print(reward_dataset.keys())


dict_keys(['train'])


In [10]:
import trl
print(trl.__version__)


0.18.1


In [2]:
import peft
print(peft.__version__)


0.15.2


In [ ]:
#!pip install --upgrade peft


In [ ]:
# !pip install tiktoken
# !pip install --upgrade protobuf


   ---------------------------------------- 0.0/893.9 kB ? eta -:--:--
   --------------------------------------- 893.9/893.9 kB 20.4 MB/s eta 0:00:00


In [ ]:
# !pip install sentencepiece


   ---------------------------------------- 0.0/991.5 kB ? eta -:--:--
   --------------------------------------- 991.5/991.5 kB 23.5 MB/s eta 0:00:00


In [ ]:
from transformers import LlamaTokenizer

try:
    tokenizer = LlamaTokenizer.from_pretrained(MODEL_ID)
except Exception as e:
    print(f"Error loading tokenizer: {e}")


Error loading tokenizer: 
LlamaTokenizer requires the SentencePiece library but it was not found in your environment. Checkout the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.



In [2]:
import os
print(os.listdir("sft_tinyllama"))


[]


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "sshleifer/tiny-gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:799: UserWarning: Not enough free disk space to download the file. The expected file size is: 0.00 MB. The target location C:\Users\gadep\.cache\huggingface\hub\models--sshleifer--tiny-gpt2\blobs only has 0.00 MB free disk space.
  warnings.warn(
C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gadep\.cache\huggingface\hub\models--sshleifer--tiny-gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environme

In [ ]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files=CSV_FILE_PATH, split="train")

batch_size = 8
for i in range(0, len(dataset), batch_size):
    batch = dataset.select(range(i, min(i+batch_size, len(dataset))))
    prompts = batch["chunk_text"] 


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" 

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)


In [14]:
inputs = tokenizer("Tell me a joke.", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Tell me a joke.

Joke: "What do you call a pig in a poke?"

Audience: "A pig in a poke!"

Joke: "What do you call a pig in a poke


## 🚀 Reinforcement Learning using REINFORCE (Instead of PPO)

In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.cuda()
    

In [ ]:

def generate_response(prompt, max_length=128):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_length=max_length, do_sample=True, top_k=50, top_p=0.95)
    return tokenizer.decode(output[0], skip_special_tokens=True)
    

In [ ]:

def compute_reward(prompt, response):
    # Dummy reward function — replace with real reward model or human feedback
    if "summary" in response.lower():
        return 1.0
    else:
        return 0.2
    

In [ ]:

def reinforce_loss(log_probs, reward):
    return -log_probs.sum() * reward
    

In [ ]:

from torch.optim import Adam

optimizer = Adam(model.parameters(), lr=5e-6)
num_epochs = 1

for epoch in range(num_epochs):
    for example in dataset["train"][:10]:  # small batch for testing
        prompt = f"<|user|> Summarize the following:\n{example['chunk_text']} <|assistant|>"
        response = generate_response(prompt)
        reward = compute_reward(prompt, response)
        
        # Re-tokenize prompt + response for log probs
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = tokenizer(prompt + response, return_tensors="pt").to(model.device)
        labels = outputs["input_ids"]
        
        logits = model(**inputs).logits
        log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
        selected_log_probs = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
        
        loss = reinforce_loss(selected_log_probs, reward)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        print(f"Prompt: {prompt[:50]}...")
        print(f"Response: {response[:50]}...")
        print(f"Reward: {reward}, Loss: {loss.item()}")
    

In [ ]:

# Save the fine-tuned model
model.save_pretrained("tinyllama_reinforce_finetuned")
tokenizer.save_pretrained("tinyllama_reinforce_finetuned")
    